# 03 — Supervised Learning & Evaluation

**Primary:** Zhilin Zhang  
**Support:** Tuan Wei

Mandatory models: **K-Nearest Neighbours (KNN)** and **Decision Tree**.

The RQ is tested with two matched feature sets:
1. **size_only** — property-size attributes only;
2. **size_location_amenities** — the same size attributes plus location and amenity features.

The exact train/test split and high-price target are loaded from preprocessing so every section uses the same rows and threshold.

## 1. Imports and data

In [1]:
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
import sklearn

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Generate data/processed_listings.csv with the preprocessing stage first.")

df = pd.read_csv(DATA_PATH)
print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())
print("Target by split:")
display(pd.crosstab(df["split"], df["high_price"], margins=True))

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}
Target by split:


high_price,0,1,All
split,,,
test,2171,724,2895
train,8683,2894,11577
All,10854,3618,14472


## 2. Feature sets

In [2]:
SIZE_FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
]

LOCATION_FEATURES = [
    "distance_cbd_km",
]

AMENITY_FEATURES = [
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
]

FEATURE_SETS = {
    "size_only": SIZE_FEATURES,
    "size_location_amenities": (
        SIZE_FEATURES + LOCATION_FEATURES + AMENITY_FEATURES
    ),
}

TARGET = "high_price"

for name, features in FEATURE_SETS.items():
    missing = [c for c in features if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

train_df = (
    df.loc[df["split"].eq("train")].sort_values("id", kind="stable").reset_index(drop=True)
)
test_df = (
    df.loc[df["split"].eq("test")].sort_values("id", kind="stable").reset_index(drop=True)
)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))
print("Train positive rate:", round(train_df[TARGET].mean(), 4))
print("Test positive rate:", round(test_df[TARGET].mean(), 4))

runtime_environment = pd.DataFrame([{
    "operating_system": platform.system(),
    "os_release": platform.release(),
    "machine": platform.machine(),
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scipy_version": scipy.__version__,
    "scikit_learn_version": sklearn.__version__,
    "knn_training_order": "listing id ascending (stable)",
    "knn_algorithm": "brute",
    "knn_n_jobs": 1,
}])
display(runtime_environment)


Train rows: 11577
Test rows: 2895
Train positive rate: 0.25
Test positive rate: 0.2501


,operating_system,os_release,machine,python_version,numpy_version,pandas_version,scipy_version,scikit_learn_version,knn_training_order,knn_algorithm,knn_n_jobs
0,Linux,6.17.0-1022-azure,x86_64,3.11.16,2.3.5,2.3.3,1.16.3,1.7.2,listing id ascending (stable),brute,1


## 3. Evaluation helpers

In [3]:
def metric_summary(y_true, y_pred, y_score=None):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],
        zero_division=0,
    )

    result = {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "precision_class_0": float(precision[0]),
        "recall_class_0": float(recall[0]),
        "f1_class_0": float(f1[0]),
        "support_class_0": int(support[0]),
        "precision_class_1": float(precision[1]),
        "recall_class_1": float(recall[1]),
        "f1_class_1": float(f1[1]),
        "support_class_1": int(support[1]),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        result["roc_auc"] = float(roc_auc_score(y_true, y_score))
    else:
        result["roc_auc"] = np.nan

    return result


def bootstrap_macro_f1_ci(
    y_true,
    y_pred,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rng = np.random.default_rng(random_state)
    n = len(y_true)
    scores = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores.append(
            f1_score(
                y_true[idx],
                y_pred[idx],
                average="macro",
                zero_division=0,
            )
        )

    scores = np.asarray(scores)
    lo, hi = np.quantile(scores, [alpha/2, 1-alpha/2])

    return {
        "bootstrap_mean_macro_f1": float(scores.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "n_boot": int(n_boot),
    }

## 4. Majority-class baseline

The baseline is evaluated on the held-out test set with the same metrics as the trained models.

In [4]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(
    np.zeros((len(train_df), 1)),
    train_df[TARGET],
)
baseline_pred = baseline.predict(np.zeros((len(test_df), 1)))
baseline_score = baseline.predict_proba(np.zeros((len(test_df), 1)))[:, 1]

baseline_metrics = metric_summary(
    test_df[TARGET].to_numpy(),
    baseline_pred,
    baseline_score,
)
baseline_metrics

{'accuracy': 0.7499136442141624,
 'macro_f1': 0.42854322937228584,
 'precision_class_0': 0.7499136442141624,
 'recall_class_0': 1.0,
 'f1_class_0': 0.8570864587445717,
 'support_class_0': 2171,
 'precision_class_1': 0.0,
 'recall_class_1': 0.0,
 'f1_class_1': 0.0,
 'support_class_1': 724,
 'roc_auc': 0.5}

## 5. Model pipelines and tuning grids

KNN uses median imputation + standardisation because it is distance-based.

Decision Tree uses median imputation but no scaling.

Both use 5-fold stratified CV on the training set and tune macro-F1. Every tried parameter combination is saved.

In [5]:
def make_knn_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        # Brute-force single-thread neighbour search plus stable ID ordering
        # makes exact-distance tie handling reproducible across tested runtimes.
        ("model", KNeighborsClassifier(algorithm="brute", n_jobs=1)),
    ])

def make_tree_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ])

KNN_GRID = {
    "model__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],
}

TREE_GRID = {
    "model__max_depth": [None, 3, 5, 8, 12],
    "model__min_samples_leaf": [1, 5, 15, 30],
}

CV = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("KNN combinations:", np.prod([len(v) for v in KNN_GRID.values()]))
print("Tree combinations:", np.prod([len(v) for v in TREE_GRID.values()]))


KNN combinations: 28
Tree combinations: 20


## 6. Tune and evaluate each model × feature-set combination

In [6]:
def run_experiment(model_name, feature_set_name):
    features = FEATURE_SETS[feature_set_name]

    X_train = train_df[features]
    y_train = train_df[TARGET]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    if model_name == "KNN":
        pipeline = make_knn_pipeline()
        grid = KNN_GRID
    elif model_name == "DecisionTree":
        pipeline = make_tree_pipeline()
        grid = TREE_GRID
    else:
        raise ValueError(model_name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        scoring="f1_macro",
        cv=CV,
        n_jobs=1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pred = search.predict(X_test)
    proba = search.predict_proba(X_test)[:, 1]

    metrics = metric_summary(y_test.to_numpy(), pred, proba)
    metrics.update({
        "model": model_name,
        "feature_set": feature_set_name,
        "best_cv_macro_f1": float(search.best_score_),
        "best_params": json.dumps(search.best_params_, sort_keys=True),
        "absolute_macro_f1_improvement_vs_baseline": (
            metrics["macro_f1"] - baseline_metrics["macro_f1"]
        ),
    })

    cv_results = pd.DataFrame(search.cv_results_)
    cv_results.insert(0, "model", model_name)
    cv_results.insert(1, "feature_set", feature_set_name)

    return {
        "search": search,
        "pred": pred,
        "proba": proba,
        "metrics": metrics,
        "cv_results": cv_results,
        "confusion_matrix": confusion_matrix(y_test, pred, labels=[0, 1]),
        "features": features,
    }

experiments = {}

for model_name in ["KNN", "DecisionTree"]:
    for feature_set_name in FEATURE_SETS:
        print("Running:", model_name, feature_set_name)
        experiments[(model_name, feature_set_name)] = run_experiment(
            model_name,
            feature_set_name,
        )

model_summary = pd.DataFrame(
    [result["metrics"] for result in experiments.values()]
)

display(
    model_summary[
        [
            "model",
            "feature_set",
            "best_cv_macro_f1",
            "accuracy",
            "macro_f1",
            "roc_auc",
            "f1_class_0",
            "f1_class_1",
            "absolute_macro_f1_improvement_vs_baseline",
            "best_params",
        ]
    ].sort_values(["model", "feature_set"])
)

Running: KNN size_only


Running: KNN size_location_amenities


Running: DecisionTree size_only


Running: DecisionTree size_location_amenities


,model,feature_set,best_cv_macro_f1,accuracy,macro_f1,roc_auc,f1_class_0,f1_class_1,absolute_macro_f1_improvement_vs_baseline,best_params
3,DecisionTree,size_location_amenities,0.753867,0.830743,0.757502,0.837520,0.890771,0.624233,0.328959,"{""model__max_depth"": 5, ""model__min_samples_le..."
2,DecisionTree,size_only,0.754895,0.824180,0.757990,0.820028,0.884554,0.631427,0.329447,"{""model__max_depth"": 3, ""model__min_samples_le..."
1,KNN,size_location_amenities,0.743824,0.817617,0.732198,0.817652,0.883444,0.580952,0.303655,"{""model__n_neighbors"": 15, ""model__p"": 1, ""mod..."
0,KNN,size_only,0.746125,0.816926,0.739680,0.813308,0.881485,0.597876,0.311137,"{""model__n_neighbors"": 21, ""model__p"": 1, ""mod..."


## 7. Hyperparameter defaults, chosen values, and specific CV effect

For each tuned parameter, this table compares the selected setting with that parameter reset to its scikit-learn default while holding the other selected parameters fixed. The difference in mean 5-fold CV macro-F1 gives a dataset-specific effect for every changed hyperparameter.

In [7]:
MODEL_DEFAULTS = {
    "KNN": {
        "model__n_neighbors": 5,
        "model__weights": "uniform",
        "model__p": 2,
    },
    "DecisionTree": {
        "model__max_depth": None,
        "model__min_samples_leaf": 1,
    },
}

hyperparameter_effect_rows = []

for (model_name, feature_set_name), exp in experiments.items():
    best_params = exp["search"].best_params_
    best_score = float(exp["search"].best_score_)
    cv_table = exp["cv_results"]

    for parameter, default_value in MODEL_DEFAULTS[model_name].items():
        chosen_value = best_params[parameter]
        changed = chosen_value != default_value

        comparison_params = dict(best_params)
        comparison_params[parameter] = default_value

        mask = cv_table["params"].apply(
            lambda p: all(p.get(k) == v for k, v in comparison_params.items())
        )

        if mask.any():
            comparison_score = float(
                cv_table.loc[mask, "mean_test_score"].iloc[0]
            )
            effect = best_score - comparison_score
        else:
            comparison_score = np.nan
            effect = np.nan

        hyperparameter_effect_rows.append({
            "model": model_name,
            "feature_set": feature_set_name,
            "parameter": parameter.replace("model__", ""),
            "default_value": str(default_value),
            "chosen_value": str(chosen_value),
            "changed_from_default": bool(changed),
            "chosen_cv_macro_f1": best_score,
            "cv_macro_f1_with_parameter_at_default": comparison_score,
            "specific_cv_macro_f1_effect": effect,
        })

hyperparameter_effects = pd.DataFrame(hyperparameter_effect_rows)
display(hyperparameter_effects)


,model,feature_set,parameter,default_value,chosen_value,changed_from_default,chosen_cv_macro_f1,cv_macro_f1_with_parameter_at_default,specific_cv_macro_f1_effect
0,KNN,size_only,n_neighbors,5,21,True,0.746125,0.713329,0.032796
1,KNN,size_only,weights,uniform,uniform,False,0.746125,0.746125,0.000000
2,KNN,size_only,p,2,1,True,0.746125,0.744439,0.001686
3,KNN,size_location_amenities,n_neighbors,5,15,True,0.743824,0.723934,0.019890
4,KNN,size_location_amenities,weights,uniform,distance,True,0.743824,0.739942,0.003882
5,KNN,size_location_amenities,p,2,1,True,0.743824,0.734143,0.009681
6,DecisionTree,size_only,max_depth,None,3,True,0.754895,0.743315,0.011580
7,DecisionTree,size_only,min_samples_leaf,1,15,True,0.754895,0.754637,0.000258
8,DecisionTree,size_location_amenities,max_depth,None,5,True,0.753867,0.739931,0.013937
9,DecisionTree,size_location_amenities,min_samples_leaf,1,30,True,0.753867,0.751949,0.001918


## 8. Incremental value of location + amenities

This table directly answers the first part of the RQ for each mandatory model by subtracting size-only performance from full-feature performance.

In [8]:
incremental_rows = []

for model_name in ["KNN", "DecisionTree"]:
    size_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_only")
    ].iloc[0]
    full_row = model_summary[
        (model_summary["model"] == model_name)
        & (model_summary["feature_set"] == "size_location_amenities")
    ].iloc[0]

    incremental_rows.append({
        "model": model_name,
        "size_only_macro_f1": float(size_row["macro_f1"]),
        "full_macro_f1": float(full_row["macro_f1"]),
        "absolute_macro_f1_change": float(
            full_row["macro_f1"] - size_row["macro_f1"]
        ),
        "size_only_accuracy": float(size_row["accuracy"]),
        "full_accuracy": float(full_row["accuracy"]),
        "absolute_accuracy_change": float(
            full_row["accuracy"] - size_row["accuracy"]
        ),
        "size_only_roc_auc": float(size_row["roc_auc"]),
        "full_roc_auc": float(full_row["roc_auc"]),
        "absolute_roc_auc_change": float(
            full_row["roc_auc"] - size_row["roc_auc"]
        ),
    })

incremental_results = pd.DataFrame(incremental_rows)
display(incremental_results)

# Follow up the strong accommodates–bedrooms correlation with an actual
# held-out sensitivity check while retaining the prespecified benchmark.
redundancy_rows = []
for model_name in ["KNN", "DecisionTree"]:
    base_exp = experiments[(model_name, "size_only")]
    base_score = base_exp["metrics"]["macro_f1"]
    for dropped in ["accommodates", "bedrooms"]:
        kept = [f for f in SIZE_FEATURES if f != dropped]
        estimator = clone(base_exp["search"].best_estimator_)
        estimator.fit(train_df[kept], train_df[TARGET])
        pred = estimator.predict(test_df[kept])
        score = f1_score(test_df[TARGET], pred, average="macro", zero_division=0)
        redundancy_rows.append({
            "model": model_name,
            "dropped_feature": dropped,
            "retained_features": " | ".join(kept),
            "original_size_only_macro_f1": float(base_score),
            "drop_one_macro_f1": float(score),
            "macro_f1_change_vs_original": float(score - base_score),
        })

redundancy_sensitivity = pd.DataFrame(redundancy_rows)
redundancy_sensitivity["analysis_role"] = (
    "Post-hoc held-out sensitivity only; not used for model selection."
)
display(redundancy_sensitivity)

# Quantify the exact-distance boundary ties that make KNN implementation
# details important for this low-cardinality size-only feature set.
size_imputer = SimpleImputer(strategy="median")
size_scaler = StandardScaler()
z_train = size_scaler.fit_transform(size_imputer.fit_transform(train_df[SIZE_FEATURES]))
z_test = size_scaler.transform(size_imputer.transform(test_df[SIZE_FEATURES]))
unique_size_combinations = int(train_df[SIZE_FEATURES].drop_duplicates().shape[0])

knn_tie_rows = []
for p_value in KNN_GRID["model__p"]:
    for k_value in KNN_GRID["model__n_neighbors"]:
        neighbour_search = NearestNeighbors(
            n_neighbors=k_value + 1,
            metric="minkowski",
            p=p_value,
            algorithm="brute",
            n_jobs=1,
        ).fit(z_train)
        distances = neighbour_search.kneighbors(z_test, return_distance=True)[0]
        boundary_tie = np.isclose(
            distances[:, k_value - 1],
            distances[:, k_value],
            rtol=1e-12,
            atol=1e-12,
        )
        knn_tie_rows.append({
            "p": p_value,
            "k": k_value,
            "train_rows": len(train_df),
            "unique_size_feature_combinations": unique_size_combinations,
            "test_boundary_tie_n": int(boundary_tie.sum()),
            "test_boundary_tie_pct": float(boundary_tie.mean() * 100),
        })

knn_tie_diagnostics = pd.DataFrame(knn_tie_rows)
display(knn_tie_diagnostics)


# Quantify the downstream effect of deriving and adding CBD distance. The
# full-model hyperparameters are held fixed, so this is an ablation rather
# than a second model-selection exercise.
distance_ablation_rows = []
for model_name in ["KNN", "DecisionTree"]:
    full_exp = experiments[(model_name, "size_location_amenities")]
    without_distance = [
        feature for feature in full_exp["features"]
        if feature != "distance_cbd_km"
    ]
    estimator = clone(full_exp["search"].best_estimator_)
    estimator.fit(train_df[without_distance], train_df[TARGET])
    pred = estimator.predict(test_df[without_distance])
    proba = estimator.predict_proba(test_df[without_distance])[:, 1]
    no_distance_metrics = metric_summary(test_df[TARGET].to_numpy(), pred, proba)
    distance_ablation_rows.append({
        "model": model_name,
        "analysis_role": "Post-hoc held-out ablation; fixed full-model hyperparameters.",
        "with_distance_macro_f1": float(full_exp["metrics"]["macro_f1"]),
        "without_distance_macro_f1": float(no_distance_metrics["macro_f1"]),
        "macro_f1_change_from_adding_distance": float(
            full_exp["metrics"]["macro_f1"] - no_distance_metrics["macro_f1"]
        ),
        "with_distance_roc_auc": float(full_exp["metrics"]["roc_auc"]),
        "without_distance_roc_auc": float(no_distance_metrics["roc_auc"]),
        "roc_auc_change_from_adding_distance": float(
            full_exp["metrics"]["roc_auc"] - no_distance_metrics["roc_auc"]
        ),
    })

distance_ablation = pd.DataFrame(distance_ablation_rows)
display(distance_ablation)

# The stable ascending-ID order is the recorded run. Reversing only the
# training-row order demonstrates the remaining KNN sensitivity caused by
# exact-distance boundary ties; this audit does not replace the recorded run.
reverse_train_df = train_df.sort_values("id", ascending=False, kind="stable")
knn_order_sensitivity_rows = []
for feature_set_name, features in FEATURE_SETS.items():
    recorded = experiments[("KNN", feature_set_name)]
    estimator = clone(recorded["search"].best_estimator_)
    estimator.fit(reverse_train_df[features], reverse_train_df[TARGET])
    reverse_pred = estimator.predict(test_df[features])
    reverse_proba = estimator.predict_proba(test_df[features])[:, 1]
    reverse_metrics = metric_summary(
        test_df[TARGET].to_numpy(), reverse_pred, reverse_proba
    )
    knn_order_sensitivity_rows.append({
        "feature_set": feature_set_name,
        "recorded_training_order": "listing id ascending",
        "audit_training_order": "listing id descending",
        "recorded_macro_f1": float(recorded["metrics"]["macro_f1"]),
        "reverse_order_macro_f1": float(reverse_metrics["macro_f1"]),
        "macro_f1_difference_reverse_minus_recorded": float(
            reverse_metrics["macro_f1"] - recorded["metrics"]["macro_f1"]
        ),
        "recorded_roc_auc": float(recorded["metrics"]["roc_auc"]),
        "reverse_order_roc_auc": float(reverse_metrics["roc_auc"]),
        "roc_auc_difference_reverse_minus_recorded": float(
            reverse_metrics["roc_auc"] - recorded["metrics"]["roc_auc"]
        ),
        "analysis_role": "Reproducibility stress test; not model selection.",
    })

knn_order_sensitivity = pd.DataFrame(knn_order_sensitivity_rows)
display(knn_order_sensitivity)


,model,size_only_macro_f1,full_macro_f1,absolute_macro_f1_change,size_only_accuracy,full_accuracy,absolute_accuracy_change,size_only_roc_auc,full_roc_auc,absolute_roc_auc_change
0,KNN,0.73968,0.732198,-0.007482,0.816926,0.817617,0.000691,0.813308,0.817652,0.004344
1,DecisionTree,0.75799,0.757502,-0.000488,0.824180,0.830743,0.006563,0.820028,0.837520,0.017493


,model,dropped_feature,retained_features,original_size_only_macro_f1,drop_one_macro_f1,macro_f1_change_vs_original,analysis_role
0,KNN,accommodates,bedrooms | beds | bathrooms,0.73968,0.758931,0.019250,Post-hoc held-out sensitivity only; not used f...
1,KNN,bedrooms,accommodates | beds | bathrooms,0.73968,0.739231,-0.000449,Post-hoc held-out sensitivity only; not used f...
2,DecisionTree,accommodates,bedrooms | beds | bathrooms,0.75799,0.757152,-0.000838,Post-hoc held-out sensitivity only; not used f...
3,DecisionTree,bedrooms,accommodates | beds | bathrooms,0.75799,0.735944,-0.022046,Post-hoc held-out sensitivity only; not used f...


,p,k,train_rows,unique_size_feature_combinations,test_boundary_tie_n,test_boundary_tie_pct
0,1,3,11577,867,2782,96.096718
1,1,5,11577,867,2787,96.269430
2,1,7,11577,867,2764,95.474957
3,1,11,11577,867,2800,96.718480
4,1,15,11577,867,2815,97.236615
5,1,21,11577,867,2799,96.683938
6,1,31,11577,867,2812,97.132988
7,2,3,11577,867,2778,95.958549
8,2,5,11577,867,2779,95.993092
9,2,7,11577,867,2772,95.751295


,model,analysis_role,with_distance_macro_f1,without_distance_macro_f1,macro_f1_change_from_adding_distance,with_distance_roc_auc,without_distance_roc_auc,roc_auc_change_from_adding_distance
0,KNN,Post-hoc held-out ablation; fixed full-model h...,0.732198,0.720919,0.011279,0.817652,0.77369,0.043962
1,DecisionTree,Post-hoc held-out ablation; fixed full-model h...,0.757502,0.747990,0.009513,0.837520,0.82744,0.010080


,feature_set,recorded_training_order,audit_training_order,recorded_macro_f1,reverse_order_macro_f1,macro_f1_difference_reverse_minus_recorded,recorded_roc_auc,reverse_order_roc_auc,roc_auc_difference_reverse_minus_recorded,analysis_role
0,size_only,listing id ascending,listing id descending,0.739680,0.742879,0.003199,0.813308,0.808426,-0.004882,Reproducibility stress test; not model selection.
1,size_location_amenities,listing id ascending,listing id descending,0.732198,0.732198,0.000000,0.817652,0.817605,-0.000046,Reproducibility stress test; not model selection.


## 9. Uncertainty quantification

The full-feature model used for its standalone macro-F1 interval is chosen by training-only cross-validation, not by the held-out test score. Paired bootstraps on the same held-out rows estimate both:

1. the macro-F1 difference between the full and size-only feature sets; and
2. the ROC-AUC difference between the same two feature sets.

These intervals directly quantify uncertainty in the research-question comparison. They are evaluation evidence, not additional model-selection criteria.


In [9]:
full_results = {
    model: experiments[(model, "size_location_amenities")]
    for model in ["KNN", "DecisionTree"]
}

uncertainty_model = max(
    full_results,
    key=lambda m: full_results[m]["metrics"]["best_cv_macro_f1"],
)
uncertainty_exp = full_results[uncertainty_model]

uncertainty = bootstrap_macro_f1_ci(
    test_df[TARGET].to_numpy(),
    uncertainty_exp["pred"],
    n_boot=2000,
    random_state=RANDOM_STATE,
)
uncertainty["model"] = uncertainty_model
uncertainty["feature_set"] = "size_location_amenities"

def paired_bootstrap_macro_f1_difference(
    y_true,
    pred_full,
    pred_size,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    pred_full = np.asarray(pred_full)
    pred_size = np.asarray(pred_size)
    observed = (
        f1_score(y_true, pred_full, average="macro", zero_division=0)
        - f1_score(y_true, pred_size, average="macro", zero_division=0)
    )
    rng = np.random.default_rng(random_state)
    differences = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), size=len(y_true))
        differences.append(
            f1_score(y_true[idx], pred_full[idx], average="macro", zero_division=0)
            - f1_score(y_true[idx], pred_size[idx], average="macro", zero_division=0)
        )
    differences = np.asarray(differences)
    lo, hi = np.quantile(differences, [alpha / 2, 1 - alpha / 2])
    return {
        "observed_macro_f1_difference_full_minus_size": float(observed),
        "bootstrap_mean_difference": float(differences.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "bootstrap_probability_full_greater_than_size": float((differences > 0).mean()),
        "n_boot": int(n_boot),
    }

paired_rows = []
for offset, model_name in enumerate(["KNN", "DecisionTree"]):
    result = paired_bootstrap_macro_f1_difference(
        test_df[TARGET].to_numpy(),
        experiments[(model_name, "size_location_amenities")]["pred"],
        experiments[(model_name, "size_only")]["pred"],
        n_boot=2000,
        random_state=RANDOM_STATE + offset,
    )
    result["model"] = model_name
    paired_rows.append(result)

paired_incremental_uncertainty = pd.DataFrame(paired_rows)
display(pd.DataFrame([uncertainty]))
display(paired_incremental_uncertainty)


def paired_bootstrap_roc_auc_difference(
    y_true,
    score_full,
    score_size,
    n_boot=2000,
    alpha=0.05,
    random_state=42,
):
    y_true = np.asarray(y_true)
    score_full = np.asarray(score_full)
    score_size = np.asarray(score_size)
    observed = roc_auc_score(y_true, score_full) - roc_auc_score(y_true, score_size)
    rng = np.random.default_rng(random_state)
    differences = []
    while len(differences) < n_boot:
        idx = rng.integers(0, len(y_true), size=len(y_true))
        if np.unique(y_true[idx]).size < 2:
            continue
        differences.append(
            roc_auc_score(y_true[idx], score_full[idx])
            - roc_auc_score(y_true[idx], score_size[idx])
        )
    differences = np.asarray(differences)
    lo, hi = np.quantile(differences, [alpha / 2, 1 - alpha / 2])
    return {
        "observed_roc_auc_difference_full_minus_size": float(observed),
        "bootstrap_mean_difference": float(differences.mean()),
        "ci_lower_95": float(lo),
        "ci_upper_95": float(hi),
        "bootstrap_probability_full_greater_than_size": float((differences > 0).mean()),
        "n_boot": int(n_boot),
    }

paired_auc_rows = []
for offset, model_name in enumerate(["KNN", "DecisionTree"]):
    result = paired_bootstrap_roc_auc_difference(
        test_df[TARGET].to_numpy(),
        experiments[(model_name, "size_location_amenities")]["proba"],
        experiments[(model_name, "size_only")]["proba"],
        n_boot=2000,
        random_state=RANDOM_STATE + 100 + offset,
    )
    result["model"] = model_name
    paired_auc_rows.append(result)

paired_auc_uncertainty = pd.DataFrame(paired_auc_rows)
display(paired_auc_uncertainty)


,bootstrap_mean_macro_f1,ci_lower_95,ci_upper_95,n_boot,model,feature_set
0,0.757433,0.737865,0.777451,2000,DecisionTree,size_location_amenities


,observed_macro_f1_difference_full_minus_size,bootstrap_mean_difference,ci_lower_95,ci_upper_95,bootstrap_probability_full_greater_than_size,n_boot,model
0,-0.007482,-0.007195,-0.023638,0.010606,0.2010,2000,KNN
1,-0.000488,-0.000460,-0.011464,0.010865,0.4595,2000,DecisionTree


,observed_roc_auc_difference_full_minus_size,bootstrap_mean_difference,ci_lower_95,ci_upper_95,bootstrap_probability_full_greater_than_size,n_boot,model
0,0.004344,0.004078,-0.011125,0.01899,0.7065,2000,KNN
1,0.017493,0.017624,0.010455,0.02516,1.0000,2000,DecisionTree


## 10. Feature influence for both models

Permutation importance is calculated on the same held-out test set for the two full-feature models, using macro-F1. This gives feature-level influence in the original feature space for both KNN and Decision Tree.

In [10]:
importance_rows = []

for model_name in ["KNN", "DecisionTree"]:
    exp = experiments[(model_name, "size_location_amenities")]
    features = exp["features"]
    X_test = test_df[features]
    y_test = test_df[TARGET]

    perm = permutation_importance(
        exp["search"].best_estimator_,
        X_test,
        y_test,
        scoring="f1_macro",
        n_repeats=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    for feature, mean_imp, std_imp in zip(
        features,
        perm.importances_mean,
        perm.importances_std,
    ):
        importance_rows.append({
            "model": model_name,
            "feature": feature,
            "permutation_importance_mean": float(mean_imp),
            "permutation_importance_std": float(std_imp),
        })

permutation_importance_table = pd.DataFrame(importance_rows)
display(
    permutation_importance_table.sort_values(
        ["model", "permutation_importance_mean"],
        ascending=[True, False],
    )
)

,model,feature,permutation_importance_mean,permutation_importance_std
13,DecisionTree,bedrooms,0.240683,0.008897
12,DecisionTree,accommodates,0.033180,0.003169
16,DecisionTree,distance_cbd_km,0.016930,0.005667
15,DecisionTree,bathrooms,0.004118,0.002488
17,DecisionTree,amenity_count,0.003953,0.002484
14,DecisionTree,beds,0.000000,0.000000
18,DecisionTree,has_pool,0.000000,0.000000
19,DecisionTree,has_free_parking,0.000000,0.000000
20,DecisionTree,has_air_conditioning,0.000000,0.000000
21,DecisionTree,has_dedicated_workspace,0.000000,0.000000


## 11. Save evaluation outputs and figures

In [11]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
FIG_OUT = REPO_ROOT / "output" / "figures"
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)

baseline_table = pd.DataFrame([{
    "model": "MajorityClassBaseline",
    "feature_set": "none",
    **baseline_metrics,
}])

baseline_table.to_csv(TABLE_OUT / "baseline_metrics.csv", index=False)
model_summary.to_csv(TABLE_OUT / "model_summary.csv", index=False)
hyperparameter_effects.to_csv(TABLE_OUT / "hyperparameter_effects.csv", index=False)
incremental_results.to_csv(TABLE_OUT / "model_incremental_value.csv", index=False)
paired_incremental_uncertainty.to_csv(TABLE_OUT / "model_incremental_bootstrap.csv", index=False)
paired_auc_uncertainty.to_csv(TABLE_OUT / "model_auc_incremental_bootstrap.csv", index=False)
distance_ablation.to_csv(TABLE_OUT / "model_distance_ablation.csv", index=False)
knn_order_sensitivity.to_csv(TABLE_OUT / "knn_order_sensitivity.csv", index=False)
runtime_environment.to_csv(TABLE_OUT / "runtime_environment.csv", index=False)
redundancy_sensitivity.to_csv(TABLE_OUT / "model_redundancy_sensitivity.csv", index=False)
knn_tie_diagnostics.to_csv(TABLE_OUT / "knn_tie_diagnostics.csv", index=False)
permutation_importance_table.to_csv(
    TABLE_OUT / "model_permutation_importance.csv",
    index=False,
)
pd.DataFrame([uncertainty]).to_csv(
    TABLE_OUT / "model_uncertainty.csv",
    index=False,
)

for (model_name, feature_set_name), exp in experiments.items():
    safe_model = model_name.lower()
    safe_set = feature_set_name.lower()

    keep_cols = [
        c for c in exp["cv_results"].columns
        if c.startswith("param_")
        or c in [
            "model",
            "feature_set",
            "mean_test_score",
            "std_test_score",
            "mean_train_score",
            "std_train_score",
            "rank_test_score",
            "params",
        ]
    ]
    exp["cv_results"][keep_cols].to_csv(
        TABLE_OUT / f"cv_{safe_model}_{safe_set}.csv",
        index=False,
    )

    cm = pd.DataFrame(
        exp["confusion_matrix"],
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )
    cm.to_csv(
        TABLE_OUT / f"confusion_{safe_model}_{safe_set}.csv"
    )

plot_df = model_summary.pivot(
    index="model",
    columns="feature_set",
    values="macro_f1",
)

fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(plot_df.index))
width = 0.36

ax.bar(
    x - width/2,
    plot_df["size_only"],
    width,
    label="Size only",
)
ax.bar(
    x + width/2,
    plot_df["size_location_amenities"],
    width,
    label="Size + location + amenities",
)
ax.axhline(
    baseline_metrics["macro_f1"],
    linestyle="--",
    linewidth=1.2,
    label="Majority baseline",
)
ax.set_xticks(x)
ax.set_xticklabels(plot_df.index)
ax.set_ylabel("Held-out macro-F1")
ax.set_title("Model performance by feature set")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_OUT / "model_macro_f1_comparison.png", dpi=200)
plt.close(fig)

for model_name in ["KNN", "DecisionTree"]:
    imp = permutation_importance_table[
        permutation_importance_table["model"] == model_name
    ].sort_values("permutation_importance_mean", ascending=True)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(
        imp["feature"],
        imp["permutation_importance_mean"],
        xerr=imp["permutation_importance_std"],
    )
    ax.set_xlabel("Permutation importance (macro-F1 decrease)")
    ax.set_title(f"{model_name} feature influence")
    fig.tight_layout()
    fig.savefig(
        FIG_OUT / f"permutation_importance_{model_name.lower()}.png",
        dpi=200,
    )
    plt.close(fig)

print("Saved model outputs to:", TABLE_OUT.relative_to(REPO_ROOT))
print("Saved model figures to:", FIG_OUT.relative_to(REPO_ROOT))

Saved model outputs to: output/tables
Saved model figures to: output/figures


## 12. Hyperparameter and reproducibility evidence checklist

For the group-written Methodology/Discussion, use the saved tables to state:
- every value tried, each tuned parameter's default, selected value, and observed validation-score effect;
- the exact recorded operating system, architecture, Python, and package versions;
- that KNN uses stable listing-ID ordering and brute-force single-thread neighbour search;
- the measured frequency of exact-distance boundary ties and the training-order stress test as KNN limitations;
- the paired-bootstrap intervals for full-minus-size-only macro-F1 and ROC-AUC;
- the post-hoc distance ablation and correlated-size-feature drop-one checks, clearly labelled as held-out sensitivity analyses rather than model selection.

Use the actual output tables rather than generic model descriptions.
